<a href="https://colab.research.google.com/github/Luiz-Frederico/pbl_fase2_Cardio_IA/blob/main/FASE_2_Diagn%C3%B3stico_Automatizado__IA_no_Estetosc%C3%B3pio_Digital.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FASE 2- Diagnóstico Automatizado: IA no Estetoscópio Digital

### Instalação do Modelo spaCy para Português

In [ ]:
!python -m spacy download pt_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 40.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# Parte 1 – Frases de Sintomas e Extração de Informações

## Extração De Sintomas e Sugestão De Diagnósticos

Objetivo:

1. Criar um arquivo .txt com 10 relatos-base de pacientes (acrescidos de 4 casos complementares de validação de negação).
2. Criar uma ontologia/mapa de conhecimento em .csv.
3. Ler os relatos do arquivo .txt.
4. Identificar expressões relacionadas a sintomas.
5. Comparar os sintomas encontrados com a ontologia.
6. Sugerir uma possível condição com base na associação mais forte encontrada.
7. Tratar negação explícita ("não sinto", "nunca senti", etc.) via análise de dependência sintática do spaCy.

---

Decisões Técnicas e Conceitos Utilizados

O código foi desenvolvido para simular um sistema básico de apoio à identificação de sintomas, utilizando uma ontologia (mapa de conhecimento) como base para as associações entre expressões e possíveis condições. Os relatos são armazenados em .txt e a ontologia em .csv, permitindo separar os dados da lógica de processamento.

Foi utilizado Pandas para estruturar e manipular os dados da ontologia e spaCy para processamento de linguagem natural, principalmente a lematização, reduzindo variações das palavras e tornando a comparação mais consistente. Também foi aplicada uma etapa de normalização textual, removendo diferenças de maiúsculas, acentuação e espaços.

A lógica de decisão utiliza pontuação por associação: uma correspondência completa entre os dois elementos da ontologia recebe maior peso que uma correspondência parcial. Essa abordagem foi escolhida para evitar que um único sintoma isolado gere automaticamente múltiplas condições. A ontologia construída contém 15 associações entre pares de sintomas e doenças, cobrindo quatro condições cardíacas: *Infarto*, *Insuficiência Cardíaca*, *Angina* e *Arritmia Cardíaca*.

Tratamento de negação via análise de dependência sintática

Além do matching textual/lematizado, o sistema incorpora detecção de negação utilizando a árvore de dependências do spaCy. Tokens cujo lema pertence ao léxico de negação (`não`, `nunca`, `jamais`, `nem`, `tampouco`) definem o "cue" de negação. O escopo é coletado recursivamente a partir do verbo regente, interrompendo-se em verbos coordenados para não propagar a negação a orações independentes.

Quando todas as ocorrências de um sintoma da ontologia caem dentro desse escopo, o sintoma é marcado como `(NEGADO)` e **não pontua** para o diagnóstico. Isso evita falsos positivos como classificar "não sinto dor no peito" como sugestivo de Infarto.

Para validar essa lógica, foram adicionados 4 relatos complementares com negação explícita, além dos 10 relatos-base. A saída do sistema destaca visualmente a separação entre os dois blocos.

O resultado final é estruturado em um DataFrame e exportado para .csv, facilitando a análise e validação dos resultados.

> Nota: A célula seguinte demonstra, de forma didática, a árvore de dependências gerada pelo spaCy para uma frase com negação.

---


######Observação:
<small> Este projeto é uma simulação acadêmica. Os resultados NÃO representam diagnóstico médico real.

In [ ]:
# ============================================================
# 1. IMPORTAÇÃO E CONFIGURAÇÃO
# ============================================================
import os, re, unicodedata
import pandas as pd
import spacy

NOME_ARQUIVO_TXT = "sintomas_pacientes.txt"
NOME_ARQUIVO_CSV = "ontologia_sintomas.csv"
NOME_ARQUIVO_RESULTADOS = "resultados_diagnostico_parte1.csv"

# ============================================================
# 2. RELATOS DOS PACIENTES
# ============================================================
# --- 10 relatos-base (escopo oficial da Parte 1) ---
frases_pacientes = [
    "Há dois dias estou com uma dor no peito intensa que piora quando faço esforço físico.",
    "Sinto cansaço constante há uma semana, mesmo depois de descansar o dia todo.",
    "Estou com uma falta de ar muito forte e dificuldade para respirar desde ontem à noite.",
    "Sinto um aperto no tórax seguido de tontura quando subo as escadas da minha casa.",
    "Relato fadiga extrema e fraqueza no corpo nos últimos três dias.",
    "Estou sentindo palpitações no coração e uma dor no peito leve que vai e volta.",
    "Apresento uma falta de ar progressiva ao caminhar curtas distâncias no bairro.",
    "Sinto cansaço constante e falta de ar quando tento dormir à noite.",
    "Tenho um aperto no tórax constante acompanhado de suor frio desde hoje cedo.",
    "Relato uma dor no peito irradiando para o braço esquerdo há cerca de duas horas."
]

# --- Casos complementares: validação do tratamento de negação ---
# (teste de análise de dependência sintática do spaCy)
frases_negacao = [
    "Não sinto dor no peito, apenas um leve cansaço nas pernas.",
    "Não tenho falta de ar nem palpitações, somente um desconforto muscular.",
    "Nunca senti aperto no tórax, apenas um cansaço ocasional.",
    "Jamais senti dor no peito ou falta de ar, somente cansaço leve."
]

frases_pacientes.extend(frases_negacao)

N_RELATOS_BASE   = 10
N_CASOS_NEGACAO  = len(frases_negacao)

# Criação do arquivo TXT
with open(NOME_ARQUIVO_TXT, "w", encoding="utf-8") as arquivo_txt:
    for frase in frases_pacientes:
        arquivo_txt.write(frase + "\n")

print(f"[OK] Arquivo '{NOME_ARQUIVO_TXT}' criado/atualizado.")
print(f"[INFO] Relatos-base: {N_RELATOS_BASE} | Casos complementares (negação): {N_CASOS_NEGACAO} | Total: {len(frases_pacientes)}")

# ============================================================
# 3. ONTOLOGIA / MAPA DE CONHECIMENTO
# ============================================================
dados_ontologia = {
    "Sintoma_1": [
        "dor no peito", "dor no peito", "dor no peito", "dor no peito", "aperto no tórax",
        "aperto no tórax", "cansaço constante", "cansaço constante", "fadiga", "falta de ar",
        "falta de ar", "falta de ar", "palpitações", "palpitações", "aperto no tórax"
    ],
    "Sintoma_2": [
        "esforço físico", "braço esquerdo", "suor frio", "tontura", "suor frio", "tontura",
        "mesmo depois de descansar", "falta de ar", "fraqueza no corpo", "dificuldade para respirar",
        "caminhar", "esforço físico", "dor no peito", "tontura", "falta de ar"
    ],
    "Doenca_Associada": [
        "Infarto", "Infarto", "Infarto", "Infarto", "Infarto", "Infarto",
        "Insuficiência Cardíaca", "Insuficiência Cardíaca", "Insuficiência Cardíaca",
        "Angina", "Angina", "Angina", "Arritmia Cardíaca", "Arritmia Cardíaca", "Angina"
    ]
}

# Criação e limpeza da Ontologia
df_ontologia_inicial = pd.DataFrame(dados_ontologia)
df_ontologia_inicial = df_ontologia_inicial.drop_duplicates(subset=["Sintoma_1", "Sintoma_2", "Doenca_Associada"]).reset_index(drop=True)
df_ontologia_inicial.to_csv(NOME_ARQUIVO_CSV, index=False, encoding="utf-8")

print(f"[OK] Ontologia '{NOME_ARQUIVO_CSV}' criada/atualizada.")
print(f"[INFO] Quantidade de associações: {len(df_ontologia_inicial)}")

# ============================================================
# 4. CARREGAMENTO DO MODELO E DADOS
# ============================================================
try:
    nlp = spacy.load("pt_core_news_sm")
except OSError:
    print("\n[ERRO] O modelo 'pt_core_news_sm' não foi encontrado.")
    print("\nNo Google Colab, execute primeiro:")
    print("!pip install -q spacy")
    print("!python -m spacy download pt_core_news_sm")
    raise

print("[OK] Modelo spaCy em português carregado com sucesso.")

df_ontologia = pd.read_csv(NOME_ARQUIVO_CSV, encoding="utf-8")
print("[OK] Mapa de conhecimento carregado.")
print(f"[INFO] Registros da ontologia: {len(df_ontologia)}")

# Leitura do arquivo TXT
frases_lidas = []
with open(NOME_ARQUIVO_TXT, "r", encoding="utf-8") as arquivo_txt:
    for linha in arquivo_txt:
        linha_limpa = linha.strip()
        if linha_limpa:
            frases_lidas.append(linha_limpa)

print("[OK] Arquivo de relatos lido.")
print(f"[INFO] Relatos encontrados: {len(frases_lidas)} ({N_RELATOS_BASE} base + {len(frases_lidas) - N_RELATOS_BASE} de negação)")

# ============================================================
# 5. FUNÇÕES AUXILIARES E PREPARAÇÃO
# ============================================================
def normalizar_texto(texto):
    texto = str(texto).lower().strip()
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(caractere for caractere in texto if unicodedata.category(caractere) != "Mn")
    texto = re.sub(r"\s+", " ", texto)
    return texto

def lematizar_texto(texto, nlp_model):
    doc = nlp_model(texto)
    lemmas = [token.lemma_.lower() for token in doc if not token.is_punct]
    return " ".join(lemmas)

# --- Detecção de negação via análise de dependência sintática (Cap. 10) ---
CUES_NEGACAO = {"não", "nunca", "jamais", "nem", "tampouco"}

def _coletar_escopo_negacao(token, tokens_negados):
    """Coleta recursivamente os tokens sob escopo de um verbo/constituinte negado.
    Para em verbos coordenados (conj VERB/AUX) para não propagar a negação
    para orações independentes ligadas por 'mas', 'porém', etc."""
    tokens_negados.add(token.i)
    for filho in token.children:
        if filho.dep_ == "conj" and filho.pos_ in ("VERB", "AUX"):
            continue
        _coletar_escopo_negacao(filho, tokens_negados)

def detectar_tokens_negados(doc):
    """Retorna o conjunto de índices de tokens sob escopo de negação.
    Aceita qualquer papel sintático para o cue (advmod, neg, cc) — o que
    importa é o lema pertencer ao léxico de negação."""
    tokens_negados = set()
    for token in doc:
        if token.lemma_.lower() in CUES_NEGACAO:
            _coletar_escopo_negacao(token.head, tokens_negados)
    return tokens_negados

def expressao_esta_negada(expressao_normalizada, tokens_norm, tokens_relevantes, tokens_negados):
    """Verifica se TODAS as ocorrências da expressão no texto estão negadas.
    Compara TEXTO NORMALIZADO (não lemas), pois lemas de palavras funcionais
    (ex.: 'no' → 'em') quebrariam a correspondência com a ontologia."""
    if not tokens_negados:
        return False

    palavras = expressao_normalizada.split()
    n = len(palavras)
    if n == 0:
        return False

    achou_negada = False
    for i in range(len(tokens_norm) - n + 1):
        if tokens_norm[i:i + n] == palavras:
            span_idx = [tokens_relevantes[j].i for j in range(i, i + n)]
            if not any(idx in tokens_negados for idx in span_idx):
                return False  # existe pelo menos uma ocorrência NÃO-negada
            achou_negada = True
    return achou_negada

# Preparação das expressões da ontologia
df_ontologia["Sintoma_1_Normalizado"] = df_ontologia["Sintoma_1"].apply(normalizar_texto)
df_ontologia["Sintoma_2_Normalizado"] = df_ontologia["Sintoma_2"].apply(normalizar_texto)
df_ontologia["Sintoma_1_Lematizado"] = df_ontologia["Sintoma_1_Normalizado"].apply(lambda texto: lematizar_texto(texto, nlp))
df_ontologia["Sintoma_2_Lematizado"] = df_ontologia["Sintoma_2_Normalizado"].apply(lambda texto: lematizar_texto(texto, nlp))

def expressao_encontrada(expressao_normalizada, expressao_lematizada, texto_normalizado, texto_lematizado):
    if expressao_normalizada in texto_normalizado:
        return True
    if expressao_lematizada in texto_lematizado:
        return True
    return False

# ============================================================
# 6. FUNÇÃO PRINCIPAL DE PROCESSAMENTO
# ============================================================
def extrair_sintomas_e_diagnosticar(texto, ontologia_df, nlp_model):
    # Processa o texto ORIGINAL com spaCy (para análise de dependência)
    doc_original = nlp_model(texto)
    tokens_relevantes = [t for t in doc_original if not t.is_punct and not t.is_space]
    # Usa TEXTO NORMALIZADO (não lemas) para casar com a ontologia token-a-token
    tokens_norm = [normalizar_texto(t.text) for t in tokens_relevantes]

    # Detecta escopo de negação via dependência sintática
    tokens_negados = detectar_tokens_negados(doc_original)

    # Caminho de matching original (inalterado)
    texto_normalizado = normalizar_texto(texto)
    texto_lematizado = lematizar_texto(texto_normalizado, nlp_model)

    sintomas_encontrados = set()
    sintomas_negados = set()
    associacoes_completas = []
    pontuacoes_por_doenca = {}
    correspondencias_parciais = []

    for _, row in ontologia_df.iterrows():
        sintoma_1, sintoma_2, doenca = row["Sintoma_1"], row["Sintoma_2"], row["Doenca_Associada"]
        s1_normalizado, s2_normalizado = row["Sintoma_1_Normalizado"], row["Sintoma_2_Normalizado"]
        s1_lematizado, s2_lematizado = row["Sintoma_1_Lematizado"], row["Sintoma_2_Lematizado"]

        match_s1 = expressao_encontrada(s1_normalizado, s1_lematizado, texto_normalizado, texto_lematizado)
        match_s2 = expressao_encontrada(s2_normalizado, s2_lematizado, texto_normalizado, texto_lematizado)

        # Verificação de negação: só se aplica quando há match
        s1_negado = match_s1 and expressao_esta_negada(s1_normalizado, tokens_norm, tokens_relevantes, tokens_negados)
        s2_negado = match_s2 and expressao_esta_negada(s2_normalizado, tokens_norm, tokens_relevantes, tokens_negados)

        if s1_negado:
            sintomas_negados.add(sintoma_1)
        if s2_negado:
            sintomas_negados.add(sintoma_2)

        # Match efetivo = encontrado E não-negado
        match_s1_efetivo = match_s1 and not s1_negado
        match_s2_efetivo = match_s2 and not s2_negado

        if match_s1_efetivo and match_s2_efetivo:
            sintomas_encontrados.add(sintoma_1)
            sintomas_encontrados.add(sintoma_2)
            associacoes_completas.append(f"{sintoma_1} + {sintoma_2} → {doenca}")
            pontuacoes_por_doenca[doenca] = max(pontuacoes_por_doenca.get(doenca, 0), 3)

        elif match_s1_efetivo or match_s2_efetivo:
            if match_s1_efetivo:
                sintomas_encontrados.add(sintoma_1)
                correspondencias_parciais.append((sintoma_1, doenca))
            if match_s2_efetivo:
                sintomas_encontrados.add(sintoma_2)
                correspondencias_parciais.append((sintoma_2, doenca))

            if doenca not in pontuacoes_por_doenca:
                pontuacoes_por_doenca[doenca] = max(pontuacoes_por_doenca.get(doenca, 0), 1)

    # Definição do diagnóstico final
    if pontuacoes_por_doenca:
        maior_pontuacao = max(pontuacoes_por_doenca.values())
        diagnosticos_com_maior_pontuacao = [doenca for doenca, pontuacao in pontuacoes_por_doenca.items() if pontuacao == maior_pontuacao]
        diagnostico_final = " / ".join(sorted(diagnosticos_com_maior_pontuacao))
        pontuacao_final = maior_pontuacao
    else:
        diagnostico_final = "Diagnóstico Indeterminado"
        pontuacao_final = 0

    # Resultado da extração — sintomas com marcação de negação
    anotacoes = []
    for s in sorted(sintomas_encontrados):
        anotacoes.append(s)
    for s in sorted(sintomas_negados):
        anotacoes.append(f"{s} (NEGADO)")

    sintomas_str = ", ".join(anotacoes) if anotacoes else "Nenhum sintoma mapeado"
    associacoes_completas = list(dict.fromkeys(associacoes_completas))
    associacoes_str = " | ".join(associacoes_completas) if associacoes_completas else "Nenhuma associação completa encontrada"

    return (sintomas_str, diagnostico_final, pontuacao_final, associacoes_str)

# ============================================================
# 7. PROCESSAMENTO E EXIBIÇÃO DOS RESULTADOS
# ============================================================
resultados = []

for idx, frase in enumerate(frases_lidas, start=1):
    # --- Separador visual entre relatos-base e casos de validação de negação ---
    if idx == N_RELATOS_BASE + 1:
        print("\n\n")
        print("#" * 70)
        print("#" + " " * 68 + "#")
        print("#   CASOS COMPLEMENTARES — VALIDAÇÃO DO TRATAMENTO DE NEGAÇÃO        #")
        print("#   (Análise de dependência sintática do spaCy)                      #")
        print("#" + " " * 68 + "#")
        print("#" * 70)

    sintomas_extraidos, diagnostico, pontuacao, associacoes = extrair_sintomas_e_diagnosticar(frase, df_ontologia, nlp)

    resultados.append({
        "ID_Paciente": idx,
        "Relato_Paciente": frase,
        "Sintomas_Extraidos": sintomas_extraidos,
        "Diagnostico_Sugerido": diagnostico,
        "Pontuacao": pontuacao,
        "Associacoes_Encontradas": associacoes
    })

    print("\n" + "=" * 70)
    print(f"PACIENTE #{idx:02d}")
    print("=" * 70)
    print(f"Relato: {frase}")
    print(f"\nSintomas identificados: {sintomas_extraidos}")
    print(f"Diagnóstico sugerido: {diagnostico}")
    print(f"Pontuação: {pontuacao}")
    print(f"Associação encontrada: {associacoes}")

# ============================================================
# 8. SALVAMENTO E RESUMO FINAL
# ============================================================
df_resultados = pd.DataFrame(resultados)
df_resultados.to_csv(NOME_ARQUIVO_RESULTADOS, index=False, encoding="utf-8")

total_relatos = len(df_resultados)
relatos_com_sintomas = df_resultados[df_resultados["Sintomas_Extraidos"] != "Nenhum sintoma mapeado"].shape[0]
relatos_com_diagnostico = df_resultados[df_resultados["Diagnostico_Sugerido"] != "Diagnóstico Indeterminado"].shape[0]

print("\n")
print("=" * 70)
print("RESUMO DO PROCESSAMENTO")
print("=" * 70)
print(f"Total de relatos processados: {total_relatos}")
print(f"Relatos com sintomas identificados: {relatos_com_sintomas}")
print(f"Relatos com diagnóstico sugerido: {relatos_com_diagnostico}")
print(f"Arquivo de resultados: {NOME_ARQUIVO_RESULTADOS}")
print("\n[OK] Processamento da Parte 1 concluído com sucesso.")
print("\n[AVISO] Este resultado é uma simulação acadêmica baseada na ontologia criada para o projeto e não representa diagnóstico médico real.")

[OK] Arquivo 'sintomas_pacientes.txt' criado/atualizado.
[INFO] Relatos-base: 10 | Casos complementares (negação): 4 | Total: 14
[OK] Ontologia 'ontologia_sintomas.csv' criada/atualizada.
[INFO] Quantidade de associações: 15
[OK] Modelo spaCy em português carregado com sucesso.
[OK] Mapa de conhecimento carregado.
[INFO] Registros da ontologia: 15
[OK] Arquivo de relatos lido.
[INFO] Relatos encontrados: 14 (10 base + 4 de negação)

PACIENTE #01
Relato: Há dois dias estou com uma dor no peito intensa que piora quando faço esforço físico.

Sintomas identificados: dor no peito, esforço físico
Diagnóstico sugerido: Infarto
Pontuação: 3
Associação encontrada: dor no peito + esforço físico → Infarto

PACIENTE #02
Relato: Sinto cansaço constante há uma semana, mesmo depois de descansar o dia todo.

Sintomas identificados: cansaço constante, mesmo depois de descansar
Diagnóstico sugerido: Insuficiência Cardíaca
Pontuação: 3
Associação encontrada: cansaço constante + mesmo depois de descansar 

### Debug: inspeção da árvore de dependências

Célula de inspeção que demonstra como o spaCy estrutura uma frase com negação e quais tokens caem no escopo do cue `"não"`.

A propagação é conservadora: a negação alcança todo o escopo do verbo regente (`sinto`), sem interromper em marcadores como `"apenas"`. Isso é inofensivo porque, no pipeline principal, apenas sintomas presentes na ontologia são testados contra o escopo — mas vale registrar como direção para refinamento futuro.

In [ ]:
doc = nlp("Não sinto dor no peito, apenas um leve cansaço nas pernas.")
for t in doc:
    print(f"{t.text:12} | lemma={t.lemma_:10} | dep={t.dep_:10} | pos={t.pos_:8} | head={t.head.text}")
print("\nTokens negados:", detectar_tokens_negados(doc))

Não          | lemma=não        | dep=advmod     | pos=ADV      | head=sinto
sinto        | lemma=sinto      | dep=ROOT       | pos=VERB     | head=sinto
dor          | lemma=dor        | dep=obj        | pos=NOUN     | head=sinto
no           | lemma=em o       | dep=case       | pos=ADP      | head=peito
peito        | lemma=peito      | dep=nmod       | pos=NOUN     | head=dor
,            | lemma=,          | dep=punct      | pos=PUNCT    | head=cansaço
apenas       | lemma=apenas     | dep=advmod     | pos=ADV      | head=cansaço
um           | lemma=um         | dep=det        | pos=DET      | head=cansaço
leve         | lemma=leve       | dep=amod       | pos=ADJ      | head=cansaço
cansaço      | lemma=cansaço    | dep=obj        | pos=NOUN     | head=sinto
nas          | lemma=em o       | dep=case       | pos=ADP      | head=pernas
pernas       | lemma=perna      | dep=nmod       | pos=NOUN     | head=cansaço
.            | lemma=.          | dep=punct      | pos=PUNCT    | h

# Parte 2 – Classificador básico de texto

Esta etapa constrói um **classificador supervisionado de risco** a partir dos relatos de pacientes. O objetivo é reproduzir, com ferramentas acessíveis, a lógica por trás de sistemas de triagem clínica que ajudam a priorizar atendimentos com base na gravidade relatada.

A Parte 2 está organizada em três blocos:

1. **Pré-processamento de negação** — reutiliza a análise de dependência sintática do spaCy para marcar tokens negados antes da vetorização.
2. **Pipeline de classificação** — TF-IDF + Regressão Logística, com avaliação por acurácia, matriz de confusão e classification report.
3. **Auditoria e governança de dados** — análise de pesos, investigação de erros e ajuste do dataset para reduzir distorções, alinhada ao tema de IA responsável.

Cada bloco é detalhado nas células seguintes.

### Pré-processamento de negação com spaCy (integração com a Parte 1)

<small> Antes de vetorizar os textos com TF-IDF, aplicamos um **pré-processamento de negação** que reutiliza a mesma lógica de dependência sintática do spaCy implementada na Parte 1 .

<small> Cada token dentro do escopo de uma negação (`não`, `nunca`, `jamais`, `nem`, `tampouco`) é prefixado com `NEG_`. Assim, `dor` e `NEG_dor` se tornam **tokens distintos** para o TF-IDF, e o modelo pode aprender pesos separados para sintomas afirmados e negados.

<small> **Motivação:** TF-IDF é uma representação *bag-of-words* — não entende negação. Sem esse pré-processamento, frases como `"não sinto dor no peito"` seriam classificadas como alto risco por causa da presença de `"dor no peito"`. Com o prefixo, o modelo consegue diferenciar.


In [ ]:
# ==================================================
# PRÉ-PROCESSAMENTO DE NEGAÇÃO PARA O CLASSIFICADOR
# ==================================================
def marcar_negacao(texto, nlp_model):
    """Pré-processa um texto marcando tokens sob escopo de negação com o prefixo NEG_.

    Reutiliza a lógica de dependência sintática do spaCy (Cap. 10) implementada
    na Parte 1. Tokens dentro do escopo de negação são prefixados com 'NEG_',
    permitindo que o TF-IDF trate, por exemplo, 'dor' e 'NEG_dor' como tokens
    distintos.

    Aplica-se tanto ao treino quanto à inferência, garantindo consistência
    entre os vetores vistos pelo modelo em treinamento e em produção.
    """
    doc = nlp_model(texto)
    tokens_negados = detectar_tokens_negados(doc)
    partes = []
    for token in doc:
        if token.i in tokens_negados and not token.is_punct and not token.is_space:
            partes.append("NEG_" + token.text)
        else:
            partes.append(token.text)
    return " ".join(partes)

# Demonstração rápida
_exemplo = "Não sinto dor no peito, apenas um leve cansaço nas pernas."
print(f"[DEMO] Original   : {_exemplo}")
print(f"[DEMO] Processado : {marcar_negacao(_exemplo, nlp)}")

[DEMO] Original   : Não sinto dor no peito, apenas um leve cansaço nas pernas.
[DEMO] Processado : NEG_Não NEG_sinto NEG_dor NEG_no NEG_peito , NEG_apenas NEG_um NEG_leve NEG_cansaço NEG_nas NEG_pernas .


# Classificação de Risco em Relatos de Pacientes (Treinamento)

### Objetivo

Esta etapa implementa um modelo de **classificação supervisionada de textos**, utilizando relatos simulados de pacientes para identificar automaticamente duas categorias de risco:

* **Alto risco**
* **Baixo risco**

O objetivo é demonstrar um fluxo básico de **Processamento de Linguagem Natural (NLP)** e **Machine Learning**, desde a preparação e validação do dataset até o treinamento, avaliação e teste do modelo com novos relatos.

---

### Métodos e conceitos utilizados

#### Dataset rotulado

##### O conjunto de dados contém **50 frases simuladas**, sendo:

* 25 classificadas como `alto risco`;
* 25 classificadas como `baixo risco`.

##### A distribuição equilibrada das classes reduz um possível desequilíbrio inicial entre as categorias utilizadas no treinamento. Entre as 50 frases, 10 são **casos de negação** (5 com negação total → baixo risco; 5 com negação parcial → alto risco), desenhadas para validar o pré-processamento de negação apresentado anteriormente.

### Separação entre treino e teste

##### Os dados são divididos em:

* **80% para treinamento**;
* **20% para teste**.

A divisão utiliza `stratify=y`, mantendo proporcionalmente as duas classes nos conjuntos de treinamento e teste.

O `random_state=42` permite reproduzir a mesma divisão dos dados durante novas execuções.

### Pré-processamento de negação

Antes da vetorização, cada frase passa pela função `marcar_negacao()`, que prefixa tokens negados com `NEG_`. Esse pré-processamento é aplicado **antes** do `train_test_split`, garantindo que treino e teste usem o mesmo vocabulário.

### TF-IDF

Os relatos dos pacientes são textos e, portanto, precisam ser convertidos para uma representação numérica antes de serem utilizados pelo algoritmo de Machine Learning.

Para isso é utilizado o **TF-IDF (Term Frequency–Inverse Document Frequency)**.

O método atribui pesos aos termos considerando sua importância dentro dos documentos do conjunto de dados.

Também são utilizados:

* **unigramas:** palavras individuais;
* **bigramas:** combinações de duas palavras consecutivas;
* conversão para letras minúsculas;
* `sublinear_tf=True` para aplicar uma escala sublinear à frequência dos termos.

O vocabulário é aprendido somente a partir dos dados de treinamento e posteriormente aplicado aos dados de teste, evitando que informações do conjunto de teste participem do processo de aprendizagem.

### Regressão Logística

A **Regressão Logística** é utilizada como algoritmo de classificação supervisionada.

Neste projeto, ela realiza uma **classificação binária**, aprendendo padrões presentes nos textos de treinamento para decidir entre:

* `alto risco`;
* `baixo risco`.

O modelo também fornece probabilidades associadas às classes, permitindo observar o grau de confiança de cada previsão.

---

## Avaliação do modelo

Após o treinamento, o modelo é avaliado utilizando os dados que não participaram do treinamento.

São utilizados:

#### **Acurácia**

Indica a proporção de previsões realizadas corretamente pelo modelo em relação ao total de exemplos avaliados.

#### **Matriz de Confusão**

Permite observar a quantidade de:

* classificações corretas de alto risco;
* classificações incorretas de alto risco;
* classificações corretas de baixo risco;
* classificações incorretas de baixo risco.

#### **Classification Report**

Apresenta métricas detalhadas por classe, incluindo:

* **Precision:** proporção das previsões de uma classe que realmente pertencem a ela;
* **Recall:** proporção dos exemplos reais de uma classe que foram corretamente identificados;
* **F1-score:** combinação entre precision e recall.

---

### Teste com novos relatos

Após a avaliação, o modelo é utilizado para classificar novas frases que não fazem parte do dataset original.

Essa etapa permite observar o comportamento do classificador diante de novos textos e verificar se os padrões aprendidos durante o treinamento conseguem ser aplicados a diferentes relatos.

Também é calculada a probabilidade associada à classe prevista, permitindo observar a confiança do modelo em cada classificação.

---

#### Verificação da qualidade do dataset

Ao final do processo, são realizadas verificações básicas de qualidade dos dados:

* quantidade total de registros;
* distribuição das classes;
* existência de registros duplicados;
* existência de valores ausentes.

Essa verificação complementa a etapa de preparação dos dados e ajuda a identificar problemas que poderiam afetar o treinamento do modelo.

---

#### Fluxo do processo

**Dataset rotulado → Pré-processamento de negação → Separação treino/teste → TF-IDF → Regressão Logística → Previsões → Avaliação → Teste com novos relatos → Verificação do dataset**

##### Propósito da célula

Esta célula concentra o **pipeline principal da Parte 2**, permitindo demonstrar de forma prática o processo de construção de um classificador textual supervisionado, desde a preparação da base até a análise inicial de seu comportamento.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ==============================================================================
# 1. PREPARAÇÃO DA BASE DE DADOS ROTULADA (.csv)
# ==============================================================================
NOME_BASE_CSV = "base_triagem_risco.csv"

# Dataset simulado: 20 alto risco + 20 baixo risco + 5 negação total (baixo) + 5 negação parcial (alto)
dados_triagem = {
    "frase": [
        # --- Alto risco (20 originais) ---
        "Sinto uma dor intensa no peito que se espalha para o braço esquerdo",
        "Estou com uma pressão muito forte no tórax acompanhada de falta de ar",
        "Meu coração está disparando e tive tontura seguida de desmaio",
        "A falta de ar começou de repente e veio acompanhada de dor no peito",
        "Estou sentindo dor forte no tórax junto com suor frio e náusea",
        "Tenho uma pressão intensa no peito e formigamento no braço",
        "Sinto um aperto muito forte no peito e dificuldade para respirar",
        "Uma dor no peito começou durante o esforço físico e está muito intensa",
        "Estou com dificuldade grave para respirar e uma dor torácica súbita",
        "Tive uma tontura intensa acompanhada de dor no peito e visão turva",
        "Sinto uma pressão no peito que não melhora e estou com muita falta de ar",
        "Comecei a sentir uma dor forte no tórax acompanhada de fraqueza repentina",
        "Estou com dor intensa no peito e sensação de desmaio",
        "Sinto uma dor no peito que piora quando faço esforço e vem acompanhada de suor frio",
        "Minha respiração ficou muito difícil de repente e estou sentindo pressão no peito",
        "Tenho palpitações muito fortes acompanhadas de tontura e fraqueza",
        "Estou sentindo um aperto intenso no tórax e uma sensação de falta de ar",
        "A dor no peito é forte e começou junto com suor frio e enjoo",
        "Sinto dor torácica intensa acompanhada de falta de ar e tontura",
        "Uma forte pressão no peito começou repentinamente e estou com dificuldade para respirar",
        # --- Baixo risco (20 originais) ---
        "Estou cansado depois de um dia inteiro trabalhando",
        "Sinto uma dor nas costas depois de passar muitas horas sentado",
        "Tenho um desconforto muscular no peito depois de fazer exercícios",
        "Fico com sono e fadiga no período da tarde",
        "Estou com uma dor de cabeça que começou depois do almoço",
        "Sinto um pequeno incômodo nas articulações ao acordar",
        "Tenho sentido cansaço durante as últimas semanas, sem outros sintomas",
        "Estou com coriza e uma irritação leve na garganta",
        "Sinto um desconforto no estômago depois das refeições",
        "Tenho uma pontada na costela que desaparece rapidamente",
        "Estou cansado após caminhar por bastante tempo, mas consigo continuar normalmente",
        "Minha garganta está um pouco irritada desde ontem",
        "Sinto dores musculares depois de realizar atividades físicas",
        "Tenho um leve desconforto nas costas quando permaneço muito tempo sentado",
        "Estou com sono durante o dia porque dormi poucas horas na noite anterior",
        "Sinto uma pequena dor de cabeça que melhora quando descanso",
        "Tenho um desconforto abdominal ocasional depois de comer",
        "Estou sentindo uma dor muscular nas pernas após uma caminhada longa",
        "Minha garganta está irritada e estou com coriza leve",
        "Sinto um pequeno incômodo no ombro depois de carregar peso",
        # --- Negação total → baixo risco (5) ---
        "Não sinto dor no peito, apenas um leve cansaço",
        "Não tenho falta de ar nem palpitações",
        "Nunca senti aperto no tórax, somente um cansaço ocasional",
        "Jamais tive dor no peito ou desmaio",
        "Não sinto pressão no peito, apenas um desconforto leve nas costas",
        # --- Negação parcial → alto risco (5) ---
        "Não tenho falta de ar, mas sinto uma dor no peito intensa",
        "Não sinto tontura, porém tenho dor no peito que irradia para o braço",
        "Não tenho palpitações, mas estou com aperto no tórax e suor frio",
        "Não estou com desmaio, mas sinto dor torácica forte e falta de ar",
        "Não sinto fraqueza, porém tenho pressão no peito e dificuldade para respirar"
    ],
    "situacao": (
        ["alto risco"] * 20 +
        ["baixo risco"] * 20 +
        ["baixo risco"] * 5 +
        ["alto risco"] * 5
    )
}

# Criação/atualização incondicional da base
df_base = pd.DataFrame(dados_triagem)
df_base.to_csv(NOME_BASE_CSV, index=False, encoding="utf-8")
print(f"[OK] Base rotulada '{NOME_BASE_CSV}' criada/atualizada com {len(df_base)} frases.")

[OK] Base rotulada 'base_triagem_risco.csv' criada/atualizada com 50 frases.


### Ingestão e pré-processamento

A base rotulada é carregada do `.csv` e cada frase passa por `marcar_negacao()` **antes** da divisão treino/teste. Isso garante que o modelo veja, em treino e em teste, exatamente a mesma representação (tokens negados prefixados com `NEG_`).

A divisão estratificada mantém a proporção entre as classes nos dois conjuntos.

In [ ]:
# ==============================================================================
# 2. INGESTÃO E DIVISÃO DOS DADOS (TREINO E TESTE)
# ==============================================================================
df = pd.read_csv(NOME_BASE_CSV, encoding="utf-8")

# Pré-processamento de negação: aplicado ANTES do split (consistência treino/teste)
df["frase"] = df["frase"].apply(lambda t: marcar_negacao(t, nlp))

X = df["frase"]
y = df["situacao"]

# Divisão estratificada (80% treino / 20% teste)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print(f"\n[INFO] Dados carregados: {len(df)} amostras totais.")
print(f"[INFO] Treino: {len(X_train)} frases | Teste: {len(X_test)} frases.")


[INFO] Dados carregados: 50 amostras totais.
[INFO] Treino: 40 frases | Teste: 10 frases.


In [ ]:
# ==============================================================================
# 3. VETORIZAÇÃO E TREINAMENTO DO MODELO
# ==============================================================================
vectorizer = TfidfVectorizer(ngram_range=(1, 2), lowercase=True, sublinear_tf=True)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

modelo = LogisticRegression(random_state=42)
modelo.fit(X_train_tfidf, y_train)
print("[OK] Modelo treinado com sucesso.")

# ==============================================================================
# 4. AVALIAÇÃO E MÉTRICAS DE DESEMPENHO
# ==============================================================================
y_pred = modelo.predict(X_test_tfidf)
acuracia = accuracy_score(y_test, y_pred)
matriz_confusao = confusion_matrix(y_test, y_pred, labels=["alto risco", "baixo risco"])

print("\n" + "=" * 80)
print("RELATÓRIO DE AVALIAÇÃO DO MODELO DE TRIAGEM")
print("=" * 80)
print(f"• Acurácia Global: {acuracia * 100:.2f}%\n")

print("• Matriz de Confusão:")
print("                   Predito: Alto | Predito: Baixo")
print(f"  Real: Alto Risco    {matriz_confusao[0][0]:^11} | {matriz_confusao[0][1]:^14}")
print(f"  Real: Baixo Risco   {matriz_confusao[1][0]:^11} | {matriz_confusao[1][1]:^14}\n")

print("• Relatório Detalhado por Classe:")
print(classification_report(y_test, y_pred))

# ==============================================================================
# 5. TESTE DE AUDITORIA E PREDIÇÃO EM NOVAS FRASES
# ==============================================================================
# As frases de teste também passam por `marcar_negacao()` antes de serem vetorizadas.
# Isso garante que o modelo as veja no mesmo formato com que foi treinado.
# Caso contrário, ele encontraria tokens desconhecidos (`dor`, `peito` sem `NEG_`) e classificaria incorretamente.

novas_frases_pacientes = [
    "Sinto um aperto no peito muito forte e suor frio há meia hora.",
    "Tive um leve cansaço nas pernas depois de caminhar bastante.",
    "Estou com dor de cabeça e garganta inflamada.",
    "Sinto falta de ar intensa ao ficar em pé e dor torácica.",
    # Novos casos de negação para auditoria
    "Não sinto dor no peito, apenas um cansaço leve.",
    "Não tenho falta de ar, mas sinto dor no peito forte."
]

print("=" * 80)
print("TESTE EM NOVOS RELATOS DE PACIENTES")
print("=" * 80)

# Pré-processamento de negação nos novos relatos
novas_frases_processadas = [marcar_negacao(f, nlp) for f in novas_frases_pacientes]

novos_vetores = vectorizer.transform(novas_frases_processadas)
novas_predicoes = modelo.predict(novos_vetores)
probabilidades = modelo.predict_proba(novos_vetores)
classes = modelo.classes_

for frase, pred, prob in zip(novas_frases_pacientes, novas_predicoes, probabilidades):
    idx_classe = list(classes).index(pred)
    confianca = prob[idx_classe] * 100
    print(f'Relato       : "{frase}"')
    print(f"Classificação: [{pred.upper()}] (Confiança: {confianca:.1f}%)")
    print("-" * 80)

# ==============================================================================
# 6. VERIFICAÇÃO DA QUALIDADE DO DATASET
# ==============================================================================
print("\n" + "=" * 80)
print("VERIFICAÇÃO DO DATASET")
print("=" * 80)
print(f"• Total de registros: {len(df)}")
print("\n• Distribuição das classes:")
print(df["situacao"].value_counts())
print(f"\n• Registros duplicados: {df.duplicated().sum()}")
print(f"• Valores ausentes:")
print(df.isnull().sum())


[OK] Modelo treinado com sucesso.

RELATÓRIO DE AVALIAÇÃO DO MODELO DE TRIAGEM
• Acurácia Global: 90.00%

• Matriz de Confusão:
                   Predito: Alto | Predito: Baixo
  Real: Alto Risco         5      |       0       
  Real: Baixo Risco        1      |       4       

• Relatório Detalhado por Classe:
              precision    recall  f1-score   support

  alto risco       0.83      1.00      0.91         5
 baixo risco       1.00      0.80      0.89         5

    accuracy                           0.90        10
   macro avg       0.92      0.90      0.90        10
weighted avg       0.92      0.90      0.90        10

TESTE EM NOVOS RELATOS DE PACIENTES
Relato       : "Sinto um aperto no peito muito forte e suor frio há meia hora."
Classificação: [ALTO RISCO] (Confiança: 58.6%)
--------------------------------------------------------------------------------
Relato       : "Tive um leve cansaço nas pernas depois de caminhar bastante."
Classificação: [BAIXO RISCO] (Confia

# Análise dos Pesos da Regressão Logística

Nesta etapa, examinamos os **pesos atribuídos pela Regressão Logística aos termos do TF-IDF** para entender quais palavras e expressões influenciam as decisões do modelo.

O objetivo é verificar se o modelo está aprendendo **padrões coerentes com as classes de risco** ou se existem **distorções causadas pela composição do dataset** — por exemplo, termos ambíguos com peso excessivo ou prefixos `NEG_` ainda pouco representados.

Em vez de buscar imediatamente aumentar a acurácia, optamos primeiro por **auditar o comportamento do modelo**. Essa abordagem permite identificar relações artificiais aprendidas antes de ajustar algoritmo ou hiperparâmetros.

In [ ]:
# ==============================================================================
# 7. ANÁLISE DOS PESOS DA REGRESSÃO LOGÍSTICA
# ==============================================================================
termos = vectorizer.get_feature_names_out()
coeficientes = modelo.coef_[0]

df_pesos = pd.DataFrame({"termo": termos, "peso": coeficientes})
df_pesos_ordenado = df_pesos.sort_values(by="peso", ascending=False)

print("\n" + "=" * 80)
print("ANÁLISE DOS TERMOS QUE INFLUENCIAM A CLASSIFICAÇÃO")
print("=" * 80)

# Termos com maior peso positivo (baixo risco)
print("\n→ Termos que mais favorecem a classe 'baixo risco':")
print(df_pesos_ordenado[["termo", "peso"]].head(15).to_string(index=False))

# Termos com maior peso negativo (alto risco)
print("\n→ Termos que mais favorecem a classe 'alto risco':")
print(df_pesos_ordenado[["termo", "peso"]].tail(15).sort_values(by="peso").to_string(index=False))

# ==============================================================================
# 8. VISUALIZAÇÃO DOS TERMOS MAIS INFLUENTES
# ==============================================================================
print("\n" + "=" * 80)
print("INTERPRETAÇÃO")
print("=" * 80)
print("""
Na Regressão Logística:

• Peso positivo  → favorece a classe representada pelo coeficiente positivo.
• Peso negativo  → favorece a outra classe.
• Quanto maior o valor absoluto do peso, maior a influência daquele termo
  na decisão do modelo.

A análise permite identificar quais palavras e expressões aprendidas pelo
TF-IDF estão influenciando as classificações realizadas pelo modelo.
""")


ANÁLISE DOS TERMOS QUE INFLUENCIAM A CLASSIFICAÇÃO

→ Termos que mais favorecem a classe 'baixo risco':
         termo     peso
        depois 0.447380
     depois de 0.349608
            um 0.309498
           nas 0.279485
          leve 0.237141
   desconforto 0.233852
      sinto um 0.232853
      garganta 0.185609
    com coriza 0.185609
        coriza 0.185609
        cabeça 0.183721
    cabeça que 0.183721
        dor de 0.183721
     de cabeça 0.183721
um desconforto 0.182036

→ Termos que mais favorecem a classe 'alto risco':
         termo      peso
      no peito -0.458940
         peito -0.458940
            no -0.427450
       intensa -0.400872
           dor -0.370193
        dor no -0.363841
   acompanhada -0.358242
acompanhada de -0.358242
         forte -0.329988
         falta -0.318464
         de ar -0.318464
            ar -0.318464
      falta de -0.318464
       tontura -0.311442
          para -0.295007

INTERPRETAÇÃO

Na Regressão Logística:

• Peso positivo  →

# Análise dos Erros de Classificação

Após auditar os pesos, a próxima decisão é investigar **quais registros foram classificados incorretamente** pelo modelo.

A análise dos erros permite relacionar o comportamento observado nos pesos com as previsões realizadas, identificando possíveis padrões que estejam causando decisões inadequadas.

Neste momento, optamos por **não alterar o algoritmo nem buscar aumentar a acurácia imediatamente**. Primeiro analisamos os erros para verificar se estão relacionados a:

- pouca quantidade de dados;
- semelhança entre frases de classes diferentes;
- presença de termos ambíguos;
- ou padrões artificiais existentes no dataset.

Essa abordagem permite melhorar a qualidade dos dados e a capacidade de generalização do modelo de forma fundamentada, em vez de apenas otimizar uma métrica.

> **Objetivo da análise:** compreender o comportamento do modelo antes de realizar ajustes no dataset ou nos parâmetros do classificador.

In [ ]:
# ==============================================================================
# 9. ANÁLISE DOS ERROS DE CLASSIFICAÇÃO
# ==============================================================================
print("\n" + "=" * 80)
print("ANÁLISE DOS ERROS DE CLASSIFICAÇÃO")
print("=" * 80)

# Tabela com dados reais e previsões do modelo
df_analise_erros = pd.DataFrame({
    "frase": X_test.values,
    "classe_real": y_test.values,
    "classe_predita": y_pred
})

# Identificação das classificações incorretas
erros = df_analise_erros[df_analise_erros["classe_real"] != df_analise_erros["classe_predita"]]

print(f"\n[INFO] Total de exemplos avaliados: {len(df_analise_erros)}")
print(f"[INFO] Total de classificações incorretas: {len(erros)}")

if len(erros) == 0:
    print("\n[OK] Nenhum erro de classificação foi encontrado no conjunto de teste.")
else:
    print("\n[ATENÇÃO] Exemplos classificados incorretamente:\n")
    for _, linha in erros.iterrows():
        print(f"Frase          : {linha['frase']}")
        print(f"Classe real    : {linha['classe_real']}")
        print(f"Classe prevista: {linha['classe_predita']}")
        print("-" * 80)


ANÁLISE DOS ERROS DE CLASSIFICAÇÃO

[INFO] Total de exemplos avaliados: 10
[INFO] Total de classificações incorretas: 1

[ATENÇÃO] Exemplos classificados incorretamente:

Frase          : NEG_Não NEG_tenho NEG_falta NEG_de NEG_ar NEG_nem NEG_palpitações
Classe real    : baixo risco
Classe prevista: alto risco
--------------------------------------------------------------------------------


In [ ]:
# ==============================================================================
# 10. INVESTIGAÇÃO DO FALSO POSITIVO
# ==============================================================================
print("\n" + "=" * 80)
print("INVESTIGAÇÃO DO FALSO POSITIVO")
print("=" * 80)

# Frase classificada incorretamente
frase_erro = erros.iloc[0]["frase"]
print(f"\nFrase analisada:\n\"{frase_erro}\"")

# Vetorização e extração de termos
vetor_frase = vectorizer.transform([frase_erro])
indices = vetor_frase.nonzero()[1]

analise_termos = pd.DataFrame({
    "termo": vectorizer.get_feature_names_out()[indices],
    "tfidf": vetor_frase.toarray()[0][indices],
    "peso_modelo": modelo.coef_[0][indices]
})

# Contribuição dos termos
analise_termos["contribuicao"] = analise_termos["tfidf"] * analise_termos["peso_modelo"]
analise_termos = analise_termos.sort_values(by="contribuicao", ascending=False)

print("\nTermos que mais contribuíram para a decisão:")
print(analise_termos.to_string(index=False))
print("\n" + "=" * 80)


INVESTIGAÇÃO DO FALSO POSITIVO

Frase analisada:
"NEG_Não NEG_tenho NEG_falta NEG_de NEG_ar NEG_nem NEG_palpitações"

Termos que mais contribuíram para a decisão:
              termo    tfidf  peso_modelo  contribuicao
             neg_ar 0.331664    -0.092283     -0.030607
             neg_de 0.331664    -0.092283     -0.030607
      neg_de neg_ar 0.331664    -0.092283     -0.030607
          neg_falta 0.331664    -0.092283     -0.030607
   neg_falta neg_de 0.331664    -0.092283     -0.030607
neg_tenho neg_falta 0.331664    -0.092283     -0.030607
    neg_palpitações 0.331664    -0.099672     -0.033058
  neg_não neg_tenho 0.298215    -0.172597     -0.051471
          neg_tenho 0.298215    -0.172597     -0.051471
            neg_não 0.228318    -0.250738     -0.057248



# Ajuste do Dataset e Redução de Distorções

#### Objetivo

A análise do falso positivo mostrou que o modelo classificou como **alto risco** uma frase com **negação total** ("Não tenho falta de ar nem palpitações"), apesar do prefixo `NEG_` aplicado aos tokens negados.

Isso indica que, embora o pré-processamento de negação esteja correto, o **vocabulário de tokens negados ainda é pouco representado no conjunto de treino** — o modelo aprendeu os padrões de tokens afirmados com mais força do que os negados.

#### Decisão metodológica

Em vez de substituir imediatamente o algoritmo, optamos por **melhorar a representatividade do dataset**.

O objetivo é fazer com que o modelo encontre padrões mais contextualizados, evitando depender excessivamente de palavras isoladas. Para isso, serão incluídas novas frases que apresentem diferentes contextos para termos potencialmente ambíguos, como **dor**, **cansaço** e **falta de ar**.

Essa abordagem busca melhorar a capacidade de generalização do modelo e reduzir falsos positivos causados por associações artificiais presentes em uma base pequena.

Após o ajuste, o modelo será treinado novamente e sua acurácia e matriz de confusão serão comparadas com os resultados anteriores.


> **Nota:** as frases de negação adicionadas na Seção 1 já contemplam parte do viés de "termo isolado → alto risco". Esta seção complementa o ajuste com frases contextuais para os termos "dor", "cansaço" e "falta de ar" em contextos ambíguos, ampliando também a variedade de contextos afirmativos que o modelo observa no treino.

In [ ]:
# ==============================================================================
# 11. AJUSTE DO DATASET
# ==============================================================================
novos_dados = pd.DataFrame({
    "frase": [
        "Estou com uma dor muscular nas pernas depois de uma caminhada",
        "Sinto dor muscular nos braços após realizar exercícios",
        "Tenho uma dor leve nas pernas depois de uma atividade física",
        "Estou sentindo dor muscular nas costas após carregar peso",
        "Tenho um desconforto muscular nas pernas que melhora com descanso",
        "Estou com uma dor intensa no peito acompanhada de falta de ar",
        "Sinto uma dor forte no tórax que começou repentinamente",
        "Tenho uma dor intensa no peito acompanhada de suor frio",
        "Estou com dor torácica forte e sensação de desmaio",
        "Sinto uma dor muito forte no peito durante esforço físico"
    ],
    "situacao": [
        "baixo risco", "baixo risco", "baixo risco", "baixo risco", "baixo risco",
        "alto risco", "alto risco", "alto risco", "alto risco", "alto risco"
    ]
})

# Atualização e salvamento do dataset
df_atualizada = pd.concat([df, novos_dados], ignore_index=True).drop_duplicates(subset=["frase"])
df_atualizada.to_csv(NOME_BASE_CSV, index=False, encoding="utf-8")

print("[OK] Dataset atualizado com sucesso.")
print(f"[INFO] Total de registros: {len(df_atualizada)}")
print("\n[INFO] Distribuição das classes:")
print(df_atualizada["situacao"].value_counts())

[OK] Dataset atualizado com sucesso.
[INFO] Total de registros: 60

[INFO] Distribuição das classes:
situacao
alto risco     30
baixo risco    30
Name: count, dtype: int64


# Retreinamento e Reavaliação do Modelo

Objetivo

Após a inclusão de exemplos mais diversificados no dataset, o modelo será treinado novamente para verificar se passou a reconhecer melhor o contexto das frases.

Decisão metodológica

Será mantida a mesma abordagem utilizada anteriormente — divisão treino/teste, TF-IDF e Regressão Logística — alterando inicialmente apenas os dados de treinamento.

Dessa forma, podemos comparar os resultados antes e depois da melhoria do dataset sem introduzir simultaneamente mudanças no algoritmo.

O pipeline de pré-processamento também é reaplicado integralmente: o dataset atualizado é recarregado do `.csv` e cada frase passa novamente por `marcar_negacao()` antes do `train_test_split`. Isso garante que o modelo retreinado utilize exatamente a mesma representação do modelo inicial (tokens sob negação prefixados com `NEG_`), mantendo a comparação antes × depois válida.

Serão novamente avaliados:

- acurácia;
- matriz de confusão;
- precision, recall e F1-score;
- comportamento diante de novos relatos.

O objetivo não é apenas obter uma acurácia maior, mas verificar se houve redução do falso positivo identificado anteriormente e se o modelo passou a utilizar padrões mais contextualizados — incluindo o tratamento correto de sintomas negados.

In [ ]:
# ==============================================================================
# 12. RETREINAMENTO E REAVALIAÇÃO DO MODELO
# ==============================================================================
df = pd.read_csv(NOME_BASE_CSV, encoding="utf-8")

# Pré-processamento de negação (mesma função aplicada na Seção 2)
df["frase"] = df["frase"].apply(lambda t: marcar_negacao(t, nlp))

X, y = df["frase"], df["situacao"]

# Separação e vetorização TF-IDF
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), lowercase=True, sublinear_tf=True)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Treinamento e Predição
modelo = LogisticRegression(random_state=42)
modelo.fit(X_train_tfidf, y_train)
y_pred = modelo.predict(X_test_tfidf)
acuracia = accuracy_score(y_test, y_pred)

print("\n" + "=" * 80)
print("REAVALIAÇÃO DO MODELO APÓS AJUSTE DO DATASET")
print("=" * 80)
print(f"\n• Registros utilizados: {len(df)}")
print(f"• Acurácia: {acuracia * 100:.2f}%")
print("\n• Matriz de Confusão:")
print(confusion_matrix(y_test, y_pred, labels=["alto risco", "baixo risco"]))
print("\n• Relatório de Classificação:")
print(classification_report(y_test, y_pred))


REAVALIAÇÃO DO MODELO APÓS AJUSTE DO DATASET

• Registros utilizados: 60
• Acurácia: 100.00%

• Matriz de Confusão:
[[6 0]
 [0 6]]

• Relatório de Classificação:
              precision    recall  f1-score   support

  alto risco       1.00      1.00      1.00         6
 baixo risco       1.00      1.00      1.00         6

    accuracy                           1.00        12
   macro avg       1.00      1.00      1.00        12
weighted avg       1.00      1.00      1.00        12



# Comparação dos Resultados Antes e Depois do Ajuste

Objetivo

Comparar o desempenho do modelo antes e depois da melhoria do dataset, verificando se as alterações contribuíram para reduzir os erros identificados anteriormente e se o tratamento de negação foi incorporado de forma consistente ao pipeline.

Resultado observado

O modelo passou de 90,00% para 100,00% de acurácia no conjunto de teste.

Além disso, a matriz de confusão atual não apresenta falsos positivos ou falsos negativos nos 10 exemplos avaliados.

Interpretação

O resultado indica que a inclusão de exemplos mais contextualizados — tanto no que diz respeito à ambiguidade de termos como "dor", "cansaço" e "falta de ar" quanto aos casos de negação — ajudou o modelo a diferenciar melhor situações semelhantes.

Vale destacar que esta comparação contempla duas mudanças simultâneas no pipeline:

1. a ampliação do dataset com frases contextuais e casos de negação;
2. a aplicação do pré-processamento `marcar_negacao()`, que prefixa tokens sob escopo de negação com `NEG_`, permitindo ao TF-IDF distinguir sintomas afirmados de sintomas negados.

Entretanto, devido ao tamanho reduzido do dataset e do conjunto de teste, o resultado de 100% não deve ser interpretado como garantia de desempenho em dados reais.

A próxima etapa será verificar o comportamento do modelo com novos relatos — incluindo frases negativas — e avaliar se o resultado permanece consistente fora do conjunto utilizado na avaliação.

In [ ]:
# ==============================================================================
# 13. COMPARAÇÃO ANTES × DEPOIS
# ==============================================================================
acuracia_anterior = 0.90
acuracia_atual = acuracia
melhoria = (acuracia_atual - acuracia_anterior) * 100

print("\n" + "=" * 80)
print("COMPARAÇÃO DOS RESULTADOS")
print("=" * 80)
print(f"\n• Acurácia anterior : {acuracia_anterior * 100:.2f}%")
print(f"• Acurácia atual    : {acuracia_atual * 100:.2f}%")
print(f"• Melhoria          : +{melhoria:.2f} pontos percentuais")

print("\n• Resultado da avaliação atual:")
print("  - Falsos positivos: 0")
print("  - Falsos negativos: 0")
print("  - Amostras de teste: 10")


COMPARAÇÃO DOS RESULTADOS

• Acurácia anterior : 90.00%
• Acurácia atual    : 100.00%
• Melhoria          : +10.00 pontos percentuais

• Resultado da avaliação atual:
  - Falsos positivos: 0
  - Falsos negativos: 0
  - Amostras de teste: 10


### Teste interativo

O relato digitado pelo usuário passa por `marcar_negacao()` antes de ser vetorizado — mesma função usada no treino e no teste. Isso permite testar frases com ou sem negação e observar o classificador responder de forma consistente.

**Sugestões de teste:**
- `Não sinto dor no peito, apenas um cansaço leve` → esperado: baixo risco
- `Não tenho falta de ar, mas sinto dor no peito forte` → esperado: alto risco
- `Sinto aperto no peito e suor frio` → esperado: alto risco

Frases sem negação também são aceitas normalmente — o pré-processamento apenas marca os tokens negados quando eles existem, deixando o restante do texto inalterado.

A entrada é limitada a **100 caracteres** para manter o teste objetivo e adequado à visualização na interface.

> **Importante:** esta funcionalidade é uma demonstração acadêmica de Machine Learning. A classificação não representa diagnóstico médico nem deve ser utilizada para tomada de decisão clínica.

In [ ]:
# ==============================================================================
# 14. TESTE INTERATIVO DO CLASSIFICADOR
# ==============================================================================
print("\n" + "=" * 80)
print("TESTE INTERATIVO DO CLASSIFICADOR")
print("=" * 80)

relato = input("\nDigite seu relato (máximo 100 caracteres): ").strip()

if len(relato) > 100:
    print("\n[ERRO] O relato deve possuir no máximo 100 caracteres.")
elif not relato:
    print("\n[ERRO] O relato não pode estar vazio.")
else:
    # Pré-processamento de negação (mesma função usada no treino)
    relato_processado = marcar_negacao(relato, nlp)

    vetor = vectorizer.transform([relato_processado])
    predicao = modelo.predict(vetor)[0]
    probabilidades = modelo.predict_proba(vetor)[0]
    confianca = probabilidades[list(modelo.classes_).index(predicao)] * 100

    print("\n" + "-" * 80)
    print(f"Relato       : {relato}")
    print(f"Classificação: [{predicao.upper()}]")
    print(f"Confiança    : {confianca:.1f}%")
    print("-" * 80)
    print("\n[SIMULAÇÃO ACADÊMICA] O resultado não representa um diagnóstico médico.")


TESTE INTERATIVO DO CLASSIFICADOR

Digite seu relato (máximo 100 caracteres): Não sinto dor no peito, apenas um cansaço leve

--------------------------------------------------------------------------------
Relato       : Não sinto dor no peito, apenas um cansaço leve
Classificação: [BAIXO RISCO]
Confiança    : 67.3%
--------------------------------------------------------------------------------

[SIMULAÇÃO ACADÊMICA] O resultado não representa um diagnóstico médico.


# Governança, Auditoria de Vieses e Justiça Algorítmica

## Por que este tema importa

Sistemas de IA aplicados à saúde já mostraram vieses documentados na literatura — por exemplo, algoritmos de triagem que subestimam dor em mulheres ou que têm desempenho inferior em populações minoritárias. Antes de qualquer implantação real, um sistema como o desta atividade precisaria passar por auditoria de fairness.

## O que uma auditoria de vieses exigiria

1. **Atributos protegidos** — idade, sexo, raça, região, comorbidades
2. **Métricas de paridade** — Demographic Parity, Equal Opportunity, Disparate Impact
3. **Validação com comitê de ética** — aprovação institucional
4. **Monitoramento contínuo** — o modelo pode enviesar com o tempo
5. **Transparência com o paciente** — direito à explicação

## Por que não foi implementada nesta atividade

O dataset desta atividade é **simulado** e **não contém atributos demográficos**. Sem grupos protegidos, não é possível medir disparidades reais.

Ainda assim, o pipeline construído oferece o ponto de partida: a análise de pesos (Seção 7) e a investigação de erros (Seção 9) são etapas precursoras de uma auditoria. Se o dataset fosse ampliado com metadados reais, bastaria agrupar por atributo e comparar as métricas por subgrupo.

## Boas práticas adotadas neste projeto

- **Aviso ético** em todos os outputs ("simulação acadêmica, não diagnóstico médico")
- **Transparência do modelo** — análise de pesos e explicação das decisões
- **Auditoria de erros** — investigação individual do falso positivo em vez de só otimizar acurácia
- **Documentação de limitações** — reconhecimento explícito do que o sistema não cobre

Esses são os primeiros passos de uma postura de IA responsável, alinhados ao Cap. 07 do material da fase.

# Conclusão da Atividade

## Conclusão

A atividade permitiu desenvolver duas abordagens para análise de relatos de sintomas: uma baseada em **mapa de conhecimento e extração de informações** e outra utilizando **Machine Learning para classificação de risco**. Foram aplicadas técnicas de processamento de texto, TF-IDF e Regressão Logística para transformar os relatos em informações analisáveis.

Além das duas abordagens, foi implementada uma **integração entre os dois módulos** por meio do pré-processamento de negação: a mesma lógica de dependência sintática do spaCy usada na Parte 1 foi reaproveitada como etapa de pré-processamento da Parte 2, permitindo que o classificador diferencie sintomas afirmados de sintomas negados — algo que o TF-IDF puro não faria.

## Principais Entregáveis Realizados

* Arquivo `.txt` com **10 relatos-base** de pacientes + **4 casos complementares** de validação de negação.
* Arquivo `.csv` com **mapa de conhecimento** relacionando sintomas e possíveis doenças.
* Código Python para **extração de sintomas e sugestão de diagnóstico**, com **tratamento de negação via análise de dependência sintática** (Cap. 10).
* Dataset `.csv` com **60 frases classificadas** entre alto e baixo risco (incluindo 10 casos de negação).
* Classificador utilizando **TF-IDF e Regressão Logística**, com **pré-processamento de negação** integrado.
* Avaliação do modelo com **acurácia, matriz de confusão e métricas de classificação**.
* Análise de **padrões, distorções e erros de classificação**.
* Ajuste e ampliação do dataset para melhorar o comportamento do modelo.
* Testes adicionais com **frases não utilizadas no treinamento**, incluindo frases negativas.

## Desafios Encontrados

Durante o desenvolvimento, um dos principais desafios foi identificar como o modelo diferenciava expressões semelhantes entre as classes de risco. Alguns termos isolados, como **"dor"**, influenciaram a classificação de forma inadequada, gerando um falso positivo.

Outro desafio foi lidar com a **negação**. TF-IDF é *bag-of-words* e não entende negação. A solução adotada foi reutilizar a lógica de dependência sintática do spaCy (Parte 1) como pré-processamento: tokens sob escopo de negação são prefixados com `NEG_`, tornando-se distintos para o vetorizador.

## Práticas Implementadas

Para reduzir as distorções, o dataset foi ampliado com exemplos mais contextualizados e equilibrados entre **alto risco** e **baixo risco**, incluindo casos de negação total e parcial. O pré-processamento foi aplicado de forma consistente em **treino, teste e inferência**, garantindo que o modelo veja sempre a mesma representação.

**Observação:** o projeto possui finalidade exclusivamente acadêmica e os resultados não representam diagnóstico ou orientação médica real.